# Dataset loading — one pattern for the whole team

Every notebook in the repo loads mcPHASES data the same way. Two lines of setup, one line per table.

Why a unified loader exists:

1. **Big files (heart_rate.csv = 1.9 GB) crash Colab free tier when read as CSV.** The loader uses parquet copies under `dataset_parquet/` — same data, ~10× smaller, ~10× faster, and you can ask for just the columns / participants you need.
2. **Path discovery is automatic.** Works on Colab with Drive mounted, on a local clone, or with an explicit path you pass.
3. **Column names are normalized.** All DataFrames have `participant_id` (not `id`) and lowercase columns, matching the pipeline contract.

**Prerequisite:** parquet conversion must have run once on this machine. Locally: `python scripts/convert_raw_to_parquet.py`. On Colab: same command in a cell after mounting Drive.

## 1. Setup — two lines

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from utils.dataset import setup, load_mcphases

setup()                      # mounts Drive on Colab; no-op locally
data = load_mcphases()       # auto-detects paths
print(f'csv_dir     = {data.csv_dir}')
print(f'parquet_dir = {data.parquet_dir}')

## 2. What's available

Small tables are loaded eagerly. Large tables are listed but not loaded until you ask for them.

In [ ]:
data.summary()

In [ ]:
pids = data.participants()
print(f'{len(pids)} participants')
print(f'IDs: {pids}')

## 3. Small tables — dict access

Small tables (hormones + self-report, subject info, sleep summaries, etc.) are already in memory. Use `data['<table>']`.

In [ ]:
hormones = data['hormones_and_selfreport']
print(f'shape: {hormones.shape}')
print(f'columns: {list(hormones.columns)}')
hormones.head(5)

In [ ]:
data['subject_info'].head()

## 4. Large tables — `data.load(...)` with filters

For `heart_rate`, `calories`, `wrist_temperature`, `distance`, `steps`, `glucose`, `sleep`, `heart_rate_variability_details`, `estimated_oxygen_variation`: pass `columns=[...]` to read only the columns you need, and `participant_id=N` to filter to one person. Both filters are pushed down to disk — RAM stays bounded.

**Key habit on Colab:** never call `data.load('heart_rate')` without filters. Even the parquet copy is ~230 MB; loading it whole works but you usually only need part of it.

In [ ]:
import time
t = time.perf_counter()
hr_p18 = data.load('heart_rate',
                   participant_id=18,
                   columns=['id', 'day_in_study', 'bpm'])
print(f'{len(hr_p18):,} rows in {time.perf_counter()-t:.2f}s')
hr_p18.head()

In [ ]:
# Per-day mean BPM for one participant
hr_p18.groupby('day_in_study')['bpm'].agg(['mean', 'std', 'count']).head(10)

In [ ]:
# Same pattern works for any large table
wrist_temp = data.load('wrist_temperature', participant_id=18, columns=['id', 'day_in_study', 'temperature_diff_from_baseline'])
print(f'wrist_temperature: {len(wrist_temp):,} rows')
wrist_temp.head()

## 5. Migrating from the existing preprocessing notebook

The student preprocessing notebook (`notebooks_from_students/mcPHASES_preprocessing.ipynb`) loads each CSV with custom logic and skips the big ones. Here is how to replace those ~50 lines with the unified loader:

**Old:**

```python
DRIVE_PATH = '/content/drive/MyDrive/mcphases-...-1.0.0'
DATA_DIR = Path(DRIVE_PATH)

tables = {}
large_paths = {}
for path in sorted(DATA_DIR.rglob('*.csv')):
    name = path.stem
    size_mb = path.stat().st_size / 1024**2
    if size_mb > 800:
        large_paths[name] = path
    else:
        try:
            tables[name] = pd.read_csv(path, low_memory=True)
        except MemoryError:
            large_paths[name] = path

def load_participant_from_large(path, participant_id, id_col='id'):
    chunks = []
    for chunk in pd.read_csv(path, chunksize=100_000, low_memory=True):
        subset = chunk[chunk[id_col] == participant_id]
        if len(subset) > 0:
            chunks.append(subset)
    return pd.concat(chunks) if chunks else pd.DataFrame()
```

**New:**

```python
from utils.dataset import setup, load_mcphases
setup()
data = load_mcphases()

# Equivalent of `tables`:                  data.tables
# Equivalent of `large_paths`:             data.parquet_dir.glob('*.parquet')
# Equivalent of `load_participant_from_large(...)`:
#                                         data.load(name, participant_id=N)
```

Three additional things you get for free:

- `participant_id` (not `id`) on every DataFrame
- Lowercase column names everywhere
- Column-pushdown reads on big files (orders of magnitude faster than chunked CSV)

## 6. From here

- WP2 preprocessing → `notebooks/01_wp2_preprocessing.ipynb` consumes this loader.
- WP3 endpoint construction needs `data['hormones_and_selfreport']` (LH, E3G, PdG columns).
- WP5 base classifier feature extraction iterates `data.participants()` and calls `data.load(modality, participant_id=pid, columns=[...])` per participant.
- WP6/WP7 work on the contract output files in `synthetic/v1/` (or `real/v1/` once it exists), not on the raw mcPHASES tables directly.

If `load_mcphases()` raises `FileNotFoundError`, either you have not run the conversion yet (`python scripts/convert_raw_to_parquet.py`) or the dataset is in a non-standard location. In the latter case, pass it explicitly:

```python
data = load_mcphases(data_dir='/path/to/your/dataset')
```